## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies - GM Colab GPU

There are a few things I need to point out about this project, and they have been sent to the GreatLearning help desk, Ticket: 763169

**Environment Debugging Summary**

Two separate, unrelated environment issues surfaced during setup, both stemming from this Colab instance running a newer Python (3.13) than the original notebook template stated that we should use. I think since this notebook was generated, Colab updated and the 2.2.2 is no longer able to work, I had so so many errors I had to do away with using it. Here is what happened:

1. **pandas==2.2.2 build failure**: The prebuilt wheel just failed and failed, causing a source-build failure --force-reinstall forced a rebuild. Fixed by removing the hard pandas version pin and letting pip select a compatible version. I know you wanted 2.2.2 but it just would not work. I submitted a ticket about this and was told to go forward with what I did.

2. **numpy kernel-caching mismatch**: Colab's kernel auto-loads its own numpy into memory at session start, before any pip installs run. Since Python doesn't reload already-imported modules, subsequent pip installs of a different numpy version updated files on disk but never took effect in the live kernel — causing internal mismatches between numpy's top-level module and submodules loaded fresh from disk by other packages. Again another failure over failure, about 4 hours lost on this. That is why you will see the code notes that you do. Also reported to Great Learning tech support.

In [1]:
# Installation for GPU llama-cpp-python -There is a numpy warning/error - takes about 2 minutes to run
# ignore the numpy error - we need to use this version for the rest of the items so ignore and leave this as is
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 125.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 228.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 215.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 235.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.5.3 which is incompatible.


In [2]:
# THIS TAKES 2 Minutes to load - have to wait for it - Ignore the errors
# NOTE: Colab has a new runtime forcing the new upgrade on pandas, so pandas==2.2.2 can NOT be run anymore (at least - that is my conclusion). I alerted Greatlearning team
# also langchain have to have a force-reinstall is required so will do that in the next cell (splitting it out), also removed numpy setting at this version numpy==2.3.3

!pip install huggingface_hub==0.35.3 pandas tiktoken==0.12.0 pymupdf==1.26.5 chromadb sentence-transformers -q




     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/2

In [3]:
#Because I had soooo much trouble, check on numpy here
!python -c "import numpy; import numpy.strings; print('numpy OK')"

numpy OK


In [4]:
# Installing langchain separately and forcing reinstall in quiet mode - ignore the errors

!pip install langchain==0.3.27 langchain-community==0.3.31 --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 172.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.6/111.6 kB 363.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 169.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 189.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 273.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 253.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 431.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 812.1/812.1 kB 451.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 296.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.6/472.6 kB 436.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 209.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
#re-check on numpy
!python -c "import numpy; import numpy.strings; print('numpy OK')"

numpy OK


In [6]:
#check on python - again, I incurred about 5 hours of troubleshooting this nonsense, which is why I am doing this
!python -c "from numpy import *; print('star import OK')"

star import OK


In [7]:
#one more time and review
import sys
print("numpy already in sys.modules before explicit import:", "numpy" in sys.modules)

import numpy
print("numpy version:", numpy.__version__)
print("numpy loaded from:", numpy.__file__)

numpy already in sys.modules before explicit import: True
numpy version: 2.1.3
numpy loaded from: /usr/local/lib/python3.13/dist-packages/numpy/__init__.py


*DO THE COLAB RESTART SESSION HERE!!!

In [1]:
# had so many fricken errors with langchain - need to see if all okay (v 0.3.27)
!pip show langchain

Name: langchain
Version: 0.3.27
Summary: Building applications with LLMs through composability
Home-page: 
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.13/dist-packages
Requires: langchain-core, langchain-text-splitters, langsmith, pydantic, PyYAML, requests, SQLAlchemy
Required-by: langchain-community


In [2]:
# show numpy version - again so much time wasted/spent wrestling with these fricken errors - need numpy 2.5.3
!pip show numpy

Name: numpy
Version: 2.5.3
Summary: Fundamental package for array computing in Python
Home-page: https://numpy.org
Author: Travis E. Oliphant et al.
Author-email: 
License: 
Location: /usr/local/lib/python3.13/dist-packages
Requires: 
Required-by: accelerate, access, albucore, albumentations, ale-py, arviz, astropy, autograd, bigframes, blis, blosc2, bokeh, Bottleneck, bqplot, chromadb, clarabel, cmdstanpy, contourpy, cucim-cu12, cuda-core, cudf-cu12, cufflinks, cuml-cu12, cupy-cuda12x, cuvs-cu12, cvxpy, cyipopt, dask-cuda, dask-cudf-cu12, datasets, db-dtypes, diffusers, dm-tree, dopamine_rl, esda, flax, folium, formulaic, geemap, geopandas, gradio, grain, gym, gymnasium, h5netcdf, h5py, hdbscan, highspy, holoviews, hyperopt, ImageIO, imbalanced-learn, inequality, jax, jaxlib, keras, keras-hub, langchain-community, libpysal, librosa, lightgbm, llama_cpp_python, mapclassify, matplotlib, matplotlib-venn, mgwr, missingno, mizani, ml_dtypes, mlxtend, moviepy, music21, nibabel, numba, numex

## Question Answering using LLM

In [3]:
#At last - we can start on the actual project
#Libraries for processing dataframes,text - Finally we can get started with the project
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama


#### Downloading and Loading the model

In [4]:
#As this is a public model I don't need the hugging face token - in case it asks (you will see it asks for a token but okay to ignore)
model_name = "TheBloke/Llama-2-7B-Chat-GGUF"
model_file = "llama-2-7b-chat.Q4_K_M.gguf"

model_path = hf_hub_download(repo_id=model_name, filename=model_file)

llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1,   # offload all layers to Colab's T4 GPU
    n_ctx=2048,        # context window size
    verbose=True
)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


llama-2-7b-chat.Q4_K_M.gguf:   0%|          | 0.00/4.08G [00:00<?, ?B/s]

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 1 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


In [5]:
#Need to be sure 4200 (give or take) MiB of GPU memory is being used with Python process
!nvidia-smi

Sun Sep 27 15:17:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P0             27W /   70W |    4223MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

#### Response

In [6]:
#our response function with base values
def response(query,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    llm.reset()  # clear internal cache/state before generating - otherwise - query2, query 3 won't return anything
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )
    return model_output['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [7]:
#We need to query the baseline and see what it says
query_1 = "What is the protocol for managing sepsis in a critical care unit?"

answer_1 = response(query_1)
print(answer_1)


Sepsis is a life-threatening condition that requires prompt recognition and treatment. In a critical care unit, the management of sepsis involves a coordinated approach between healthcare providers, including physicians, nurses, and respiratory therapists. Here are some key components of the protocol for managing sepsis in a critical care unit:
1. Early recognition and activation of sepsis team: The sepsis team should be activated immediately upon suspicion of sepsis, and the patient should be assessed using the Sequential Organ Failure Assessment


We see here a standard, generic reply, which is NOT from the Merck Manual.

We set the tokens at 128 and that is why it is cut off - which is fine, we don't need the entire answer for what we are doing here.

Setting temperature at 0 gives a structured, numbered answer - so it is deterministic and reproducable.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [8]:
# There will be no answer on this one. This one has two questions in one. The end of the sentence (with ?) will cause a problem with this model. Please see there will be no response here.
# Again - there will be no response - this is what we expect for this "as is"
query_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

answer_2 = response(query_2)
print(answer_2)


It is an empty response!

It is because this is a known limitation of LLaMA models without the [INST][/INST] format.


In [9]:
#so why is it empty? lets check and see why
test_output = llm(
    prompt=query_2,
    max_tokens=128,
    temperature=0,
    top_p=0.95,
    top_k=50,
    echo=True
)
print(test_output)

Llama.generate: prefix-match hit


{'id': 'cmpl-0dacb7d4-18f7-488f-b13d-11e91c42337a', 'object': 'text_completion', 'created': 1790522375, 'model': '/root/.cache/huggingface/hub/models--TheBloke--Llama-2-7B-Chat-GGUF/snapshots/191239b3e26b2882fb562ffccdd1cf0f65402adb/llama-2-7b-chat.Q4_K_M.gguf', 'choices': [{'text': 'What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?\n\n\n\n\n', 'index': 0, 'logprobs': None, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 35, 'completion_tokens': 5, 'total_tokens': 40}}


Scroll to the right, and you will see STOP with only 5 completion tokens

So the model choose to stop. It predicted the ending token right away.

We need to use [INST][/INST] to stop this behaviour

In [10]:
#so add in the [INST][/INST] and it will work now.
formatted_prompt = f"[INST] {query_2} [/INST]"

test_output = llm(
    prompt=formatted_prompt,
    max_tokens=128,
    temperature=0,
    top_p=0.95,
    top_k=50,
    echo=False
)
print(test_output['choices'][0]['text'])
print(test_output['choices'][0]['finish_reason'])

Llama.generate: prefix-match hit


  Appendicitis is a medical emergency that occurs when the appendix, a small pouch-like organ located in the lower right abdomen, becomes inflamed or infected. Here are some common symptoms of appendicitis:
1. Abdominal pain: The most common symptom of appendicitis is severe pain in the lower right abdomen that can migrate to the upper right abdomen. The pain usually starts gradually and then worsens over time.
2. Nausea and vomiting: Patients with appendicitis often experience nausea and vomiting, which
length


So wrapping the prompt in LLaMA-2-Chat's expected `[INST] ... [/INST]` instruction format resolved the empty response. This is a formatting issue.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [11]:
query_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

answer_3 = response(query_3)
print(answer_3)



Sudden patchy hair loss, also known as alopecia areata, can be a distressing condition that affects both men and women. It is characterized by small, round or oval-shaped bald patches on the scalp, which may appear suddenly without any prior warning signs. The good news is that in most cases, hair growth returns spontaneously within a few months to a year, but in some instances, it can persist for longer periods.
Causes of Sudden Patchy Hair Loss:
While the exact cause of sudden patchy hair


This prompt worked fine and returned the answer. There is no break '?' in this prompt.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [12]:
query_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"

answer_4 = response(query_4)
print(answer_4)


The treatment options for a person who has sustained a physical injury to the brain tissue can vary depending on the severity and location of the injury. Here are some common treatments recommended for different types of brain injuries:
1. Concussions and mild traumatic brain injuries:
a. Rest and avoidance of activities that may exacerbate symptoms, such as sports or heavy lifting.
b. Medications to manage symptoms such as headaches, dizziness, or nausea.
c. Cognitive therapy to improve memory, attention, and other


Worked well with injury types being separated. As it is one sentence in the question, we are good.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [13]:
# Once again this will have no answer "as is" - again a limitation. Run as is and then we will fix it
query_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"

answer_5 = response(query_5)
print(answer_5)

Once again, we have a null answer like in Query 2. So lets add in the [INST][/INST] fix and see if this resolves it.

In [14]:
#so add in the [INST][/INST] and it will work now.
formatted_prompt = f"[INST] {query_5} [/INST]"

test_output = llm(
    prompt=formatted_prompt,
    max_tokens=128,
    temperature=0,
    top_p=0.95,
    top_k=50,
    echo=False
)
print(test_output['choices'][0]['text'])
print(test_output['choices'][0]['finish_reason'])

Llama.generate: prefix-match hit


  If a person fractures their leg during a hiking trip, it is essential to take prompt and appropriate action to ensure proper care and recovery. Here are some necessary precautions and treatment steps:
Precautions:
1. Stop activity: Immediately stop any physical activity and assess the injury. If the person can walk, help them move to a safe location.
2. Call for help: Contact emergency services or a park ranger for assistance. Provide as much information as possible, including the location of the incident, the severity of the injury, and any medical conditions the person may
length


In [15]:
#so why is query 5 empty? lets check and see why
test_output = llm(
    prompt=query_5,
    max_tokens=128,
    temperature=0,
    top_p=0.95,
    top_k=50,
    echo=True
)
print(test_output)

Llama.generate: prefix-match hit


{'id': 'cmpl-891a00b2-38c9-4656-9a38-4697b487effd', 'object': 'text_completion', 'created': 1790522529, 'model': '/root/.cache/huggingface/hub/models--TheBloke--Llama-2-7B-Chat-GGUF/snapshots/191239b3e26b2882fb562ffccdd1cf0f65402adb/llama-2-7b-chat.Q4_K_M.gguf', 'choices': [{'text': 'What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?\n\n\n\n\n\n\n', 'index': 0, 'logprobs': None, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 39, 'completion_tokens': 7, 'total_tokens': 46}}


The sentence in query 5, like query 2, is not a simple "give me an answer" prompt. This is why those two came back empty until we fixed it.

Now we have these for our baselines:

Query 1 (sepsis) — worked and replied

Query 2 (appendicitis) — empty, needed to add [INST][/INST] worked and replied

Query 3 (hair loss) — worked and replied

Query 4 (brain injury) — worked and replied

Query 5 (leg fracture) — empty, needed [INST][/INST] worked and replied

All the answers, including the ones with the fix, show the pre-trained knowledge, and nothing from the Merck Manual.

Everything was capped at 128 tokens so we did not get a textbook sized reply.

We will use these baselines against the RAG later.



## Question Answering using LLM with Prompt Engineering

### Query 1: What is the protocol for managing sepsis in a critical care unit?

For Query 1 - What I am going to do here is to take a deep dive here in this query, and then use my best result (tokens, temperature, et al) for the other queries.

In [16]:
#Lets have a look at all the different types, takes about a minute to run
query_1 = "What is the protocol for managing sepsis in a critical care unit?"

combinations = [
    {
        "label": "1. Raw prompt (baseline)",
        "prompt": query_1,
        "params": dict(max_tokens=128, temperature=0, top_p=0.95, top_k=50)
    },
    {
        "label": "2. [INST] template only",
        "prompt": f"[INST] {query_1} [/INST]",
        "params": dict(max_tokens=128, temperature=0, top_p=0.95, top_k=50)
    },
    {
        "label": "3. [INST] + system role, moderate temp",
        "prompt": f"[INST] <<SYS>>\nYou are a critical care physician providing evidence-based clinical guidance.\n<</SYS>>\n{query_1} [/INST]",
        "params": dict(max_tokens=256, temperature=0.3, top_p=0.9, top_k=40)
    },
    {
        "label": "4. [INST] + explicit structure request, higher temp",
        "prompt": f"[INST] {query_1} Answer in clear numbered steps. [/INST]",
        "params": dict(max_tokens=256, temperature=0.7, top_p=0.9, top_k=40)
    },
    {
        "label": "5. [INST] + system role, low temp, longer output",
        "prompt": f"[INST] <<SYS>>\nYou are a critical care physician providing evidence-based clinical guidance.\n<</SYS>>\n{query_1} [/INST]",
        "params": dict(max_tokens=300, temperature=0.2, top_p=0.85, top_k=30)
    },
]

for combo in combinations:
    llm.reset()
    output = llm(prompt=combo["prompt"], **combo["params"])
    print(f"=== {combo['label']} ===")
    print(f"Params: {combo['params']}")
    print(output['choices'][0]['text'])
    print(f"finish_reason: {output['choices'][0]['finish_reason']}\n")

=== 1. Raw prompt (baseline) ===
Params: {'max_tokens': 128, 'temperature': 0, 'top_p': 0.95, 'top_k': 50}

Sepsis is a life-threatening condition that requires prompt recognition and treatment. In a critical care unit, the management of sepsis involves a coordinated approach between healthcare providers, including physicians, nurses, and respiratory therapists. Here are some key components of the protocol for managing sepsis in a critical care unit:
1. Early recognition and activation of sepsis team: The sepsis team should be activated immediately upon suspicion of sepsis, and the patient should be assessed using the Sequential Organ Failure Assessment
finish_reason: length

=== 2. [INST] template only ===
Params: {'max_tokens': 128, 'temperature': 0, 'top_p': 0.95, 'top_k': 50}
  The management of sepsis in a critical care unit (CCU) involves a coordinated and timely response to address the various physiological and pathophysiological changes that occur in sepsis. Here is an overview

All five of these returned answers, and there were no errors

Now lets have a look at the returns.

Combo 1 (raw baseline) — generic, correct.

Combo 2 ([INST] only) — better but still rather generic.

Combo 3 (system role, temp 0.3) — Nicer, more clinical and structured.

Combo 4 (structure request, temp 0.7) — We are given steps and really decisive, with good actionable tasks, this is the best so far.

Combo 5 (system role, temp 0.2, longer) — similar to Combo 3 and abit better, but does not compare to Combo 4.


**Combo 4 is the selected**, it is explicit, and gives good output, like you would use in a clinic.



### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [17]:
# We will be using combo 4's values here on out
best_params = dict(max_tokens=256, temperature=0.7, top_p=0.9, top_k=40)

queries = {
    "query_2": "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
}

pe_answers = {}

for key, q in queries.items():
    llm.reset()
    formatted_prompt = f"[INST] {q} Answer in clear numbered steps. [/INST]"
    output = llm(prompt=formatted_prompt, **best_params)
    pe_answers[key] = output['choices'][0]['text']
    print(f"=== {key} ===")
    print(pe_answers[key])
    print(f"finish_reason: {output['choices'][0]['finish_reason']}\n")

=== query_2 ===
  Appendicitis is a medical emergency that requires prompt attention. Here are some common symptoms of appendicitis:
1. Abdominal pain: The most common symptom of appendicitis is abdominal pain, which usually starts near the navel and then moves to the lower right side of the abdomen.
2. Nausea and vomiting: Patients with appendicitis often experience nausea and vomiting, especially in the early stages of the illness.
3. Loss of appetite: People with appendicitis may lose their appetite due to the pain and discomfort in the abdomen.
4. Fever: A high fever is common in people with appendicitis, which can range from 100°F to 102°F (37.8°C to 39°C).
5. Abdominal tenderness: The abdomen of a person with appendicitis is usually tender to the touch, especially in the lower right quadrant.
6. Abdominal swelling: Appendicitis can cause the abdomen to swell due to inflammation.
7. Di
finish_reason: length



We are using Combo 4 and the answer is structured.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [18]:
best_params = dict(max_tokens=256, temperature=0.7, top_p=0.9, top_k=40)

queries = {
    "query_3": "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
}

pe_answers = {}

for key, q in queries.items():
    llm.reset()
    formatted_prompt = f"[INST] {q} Answer in clear numbered steps. [/INST]"
    output = llm(prompt=formatted_prompt, **best_params)
    pe_answers[key] = output['choices'][0]['text']
    print(f"=== {key} ===")
    print(pe_answers[key])
    print(f"finish_reason: {output['choices'][0]['finish_reason']}\n")

=== query_3 ===
  Sudden patchy hair loss, also known as alopecia areata, can be a distressing condition that causes small, round or oval-shaped bald patches on the scalp. Here are some effective treatments and solutions for addressing sudden patchy hair loss:
Step 1: Identify the underlying cause
Sudden patchy hair loss can have various causes, including:
* Autoimmune disorders (such as alopecia areata)
* Hormonal imbalances (such as thyroid disorders)
* Infections (such as fungal infections of the scalp)
* Traction alopecia (hair loss due to constant pulling or tension on the hair)
* Nutritional deficiencies (such as iron or zinc deficiency)
* Skin conditions (such as eczema or psoriasis)
Step 2: Miniaturization of Follicles
In alopecia areata, the immune system mistakenly attacks the hair follicles, causing them to shrink and eventually stop producing hair. This process is called miniaturization. Treatments that help slow down or
finish_reason: length



Tells us what is the possible cause, and we see it gives us Medications to take and Light Therapy. A good response.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [19]:
best_params = dict(max_tokens=256, temperature=0.7, top_p=0.9, top_k=40)

queries = {
    "query_4": "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
}

pe_answers = {}

for key, q in queries.items():
    llm.reset()
    formatted_prompt = f"[INST] {q} Answer in clear numbered steps. [/INST]"
    output = llm(prompt=formatted_prompt, **best_params)
    pe_answers[key] = output['choices'][0]['text']
    print(f"=== {key} ===")
    print(pe_answers[key])
    print(f"finish_reason: {output['choices'][0]['finish_reason']}\n")

=== query_4 ===
  Treatments for a person who has sustained a physical injury to brain tissue can vary depending on the severity and location of the injury. Here are some common treatments recommended for different types of brain injuries:
1. Mild Traumatic Brain Injury (TBI):
a. Rest and relaxation: A person with a mild TBI may be advised to take it easy and avoid strenuous activities for a few days or weeks to allow the brain to heal.
b. Medications: Over-the-counter pain relievers such as acetaminophen or ibuprofen can help manage headaches and other symptoms associated with mild TBI.
c. Cognitive therapy: A cognitive therapist can help a person with mild TBI improve their memory, concentration, and other cognitive functions through various exercises and techniques.
d. Vision therapy: Vision problems are common after a mild TBI, and vision therapy can help improve visual processing and perception.
2. Moderate to Severe Traumatic Brain Injury (TBI):
a. Emergency care: Immediate
finis

The answer is good, as it takes the type of injury in a stage, and gives us the progression. This was a tough question and the answer is good.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [20]:
# lowering the temperature here from 0.7 to 0.3, please see notes below as to why I did this.
best_params = dict(max_tokens=256, temperature=0.3, top_p=0.9, top_k=40)

queries = {
    "query_5": "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
}

pe_answers = {}

for key, q in queries.items():
    llm.reset()
    formatted_prompt = f"[INST] {q} Answer in clear numbered steps. [/INST]"
    output = llm(prompt=formatted_prompt, **best_params)
    pe_answers[key] = output['choices'][0]['text']
    print(f"=== {key} ===")
    print(pe_answers[key])
    print(f"finish_reason: {output['choices'][0]['finish_reason']}\n")

=== query_5 ===
  If a person has fractured their leg during a hiking trip, it is essential to take prompt and appropriate action to ensure proper care and recovery. Here are the necessary precautions and treatment steps:
Step 1: Assess the Situation and Call for Help (If Necessary)

* Check the person's condition and determine if they need immediate medical attention.
* If the fracture is severe or the person is experiencing difficulty breathing, chest pain, or other serious symptoms, call 911 or your local emergency number immediately.
Step 2: Immobilize the Injured Leg (If Possible)

* If the fracture is stable and not displaced, try to immobilize the injured leg using a splint or bandage to prevent further movement.
* Make sure the splint is securely in place and does not cause any other injuries.
Step 3: Administer First Aid (If Necessary)

* If the person is bleeding or experiencing shock, administer basic first aid measures such as applying direct pressure to the wound, elevatin

Lowering temperature from 0.7 to 0.3 for Query 5 eliminated the blank-line result I initially got. This suggests temperature=0.7 may not be uniformly optimal across all 5 queries — some benefit from more deterministic settings, particularly procedural/ factual questions like this one.

This gives a good step by step treatment plan. I would probably be somewhat critical here on if an actual hiker would be carrying ice packs with them on a hike (to me this is unrealistic), but other than that, this is a good response. I had to lower the temperature as noted.

## Data Preparation for RAG

### Loading the Data

In [21]:
#Load this first to get the google drive mounted (needs permissions - grant ok and hit continue)
from google.colab import drive
drive.mount('/content/drive')

#this is my personal data path drive - so the one for the grader of this project will be different
pdf_path = '/content/drive/MyDrive/ColabNotebooks/medical_diagnosis_manual.pdf'

loader = PyMuPDFLoader(pdf_path)
documents = loader.load()

print(f"Number of pages loaded: {len(documents)}")

Mounted at /content/drive
Number of pages loaded: 4114


### Data Overview

#### Checking the first 5 pages *(NOTE: Skipping page one as it has my email/license key)

In [23]:
#checking random pages as page 1 has my email/license key - so pages 2,3,4,5,6
for i, doc in enumerate(documents[2:7]):
    print(f"--- Page {i+2} ---")
    print(doc.page_content[:500])
    print(f"Metadata: {doc.metadata}")
    print()

--- Page 2 ---
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    .................................
Metadata: {'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'Atop CHM to PDF Converter', 'creationdate': '2012-06-15T05:44:40+00:00', 'source': '/content/drive/MyDrive/ColabNotebooks/medical_diagnosis_manual.pdf', 'file_path': '/content/drive/MyDrive/ColabNotebooks/medical_diagnosis_manual.pdf', 'total_pages': 4114, 'format': 'PDF 1.7', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'author': '', 'subject': '', 'keywords': '', 'moddate': '

#### Checking the number of pages

In [24]:
# Printing total pages and also total characters and average - may need this for chunking, but also just curious
print(f"Total number of pages: {len(documents)}")
total_chars = sum(len(doc.page_content) for doc in documents)
print(f"Total characters across all pages: {total_chars:,}")
print(f"Average characters per page: {total_chars // len(documents)}")

Total number of pages: 4114
Total characters across all pages: 13,695,434
Average characters per page: 3328


### Data Chunking

In [25]:
#Chunk the 4,114 pages using size 1000 and overlap 200
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print(f"Number of chunks created: {len(chunks)}")
print(f"Example chunk:\n{chunks[5000].page_content}")
print(f"Metadata: {chunks[5000].metadata}")

Number of chunks created: 18020
Example chunk:
• Rituximab
• Sometimes thrombopoietic-mimetic drugs
• Sometimes other immunosuppressants
• For severe bleeding, IV immune globulin, IV corticosteroids, or platelet transfusions
Adults usually are given an oral corticosteroid (eg, prednisone 1 mg/kg once/day) initially. In patients who
respond, the platelet count rises to normal within 2 to 6 wk. The corticosteroid dosage is then tapered. An
alternative corticosteroid regimen is dexamethasone 40 mg po once/day for 4 days. However, most
patients do not respond adequately or relapse as the corticosteroid is tapered. Splenectomy can achieve
a complete remission in about two thirds of these patients but is usually reserved for those with severe
thrombocytopenia, bleeding, or both; it may not be appropriate for those with mild disease. Rituximab (375
mg/m2 IV once/wk for 4 wk) may be used in patients who do not respond to splenectomy or in patients
who are not candidates for splenectomy.
Metada

**Chunking Observations**

Chunking produced 18,020 chunks from 4,114 pages using chunk_size=1000, chunk_overlap=200.

The preview confirms that chunks drawn from actual clinical content
preserve the content, with drug names and there is no uneven/odd splitting.

Limitation: since the full PDF (including front matter, table of contents, and index) was chunked without filtering, some chunks in the vector database are non-clinical noise, just as a note here

### Embedding

In [26]:
# Embed and then a sanity check
embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

# Quick sanity check: embed a sample chunk and check the vector
sample_vector = embedding_model.embed_query(chunks[5000].page_content)
print(f"Embedding vector length: {len(sample_vector)}")
print(f"First 5 values: {sample_vector[:5]}")

/tmp/ipykernel_4613/2329980922.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (3 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding vector length: 384
First 5 values: [-0.03494803234934807, -0.004230715334415436, -0.055038586258888245, -0.07353970408439636, -0.1111500933766365]


The embedding looks good with 100%.  *Just a note on the side - this was one of the primary areas where I had so many failures I had with the given/stated pandas and numpy and why I had to swtich*.

### Vector Database

In [27]:
# Make a vector DB with the data on my local drive
import shutil
import os

local_persist_directory = "/content/chroma_db"
drive_persist_directory = "/content/drive/MyDrive/chroma_db"

# Clear any old local or Drive copies to avoid duplicate/stale vectors
if os.path.exists(local_persist_directory):
    shutil.rmtree(local_persist_directory)
if os.path.exists(drive_persist_directory):
    shutil.rmtree(drive_persist_directory)

# Build fresh, locally (avoids SQLite readonly errors seen when writing directly to Drive)
vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=local_persist_directory
)

print(f"Number of vectors stored: {vectordb._collection.count()}")

# Copy the finished local database to Drive for persistence across sessions
shutil.copytree(local_persist_directory, drive_persist_directory)
print("Vector database copied to Drive for persistence.")

Number of vectors stored: 18020
Vector database copied to Drive for persistence.


Database built on my drive, takes about 2 minutes to run this.

### Retriever

In [28]:
#now to retrieve the information stored in the vector database
retriever = vectordb.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# Quick sanity check
test_results = retriever.invoke("What is the protocol for managing sepsis?")
print(f"Number of chunks retrieved: {len(test_results)}")
print(f"Top result:\n{test_results[0].page_content[:400]}")

Number of chunks retrieved: 3
Top result:
shaking chills, persistent fever, altered sensorium, hypotension, and GI symptoms (abdominal pain,
nausea, vomiting, diarrhea) suggests sepsis or septic shock. Septic shock develops in 25 to 40% of
patients with significant bacteremia.
Diagnosis
If bacteremia, sepsis, or septic shock is suspected, cultures are obtained of blood and any other
appropriate specimens (see p. 1166).
Treatment
• Antibio


### System and User Prompt Template

In [29]:
# Give the system the information it needs via these prompts to work through the questions
qna_system_message = """You are a knowledgeable medical assistant. Answer the user's question
using only the context provided below from the Merck Manual. If the context does not
contain enough information to answer confidently, say so rather than guessing."""

qna_user_message_template = """Context:
{context}

Question: {question}

Answer:"""

### Response Function

In [30]:
# The response function which takes the question and works it through the RAG with the prompts above guiding
# Going to build in a retriever as well
def generate_rag_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Build a retriever with the requested k
    dynamic_retriever = vectordb.as_retriever(search_type="similarity", search_kwargs={"k": k})
    relevant_document_chunks = dynamic_retriever.invoke(user_input)
    context_list = [d.page_content for d in relevant_document_chunks]

    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = f"[INST] {qna_system_message}\n{user_message} [/INST]"

    llm.reset()

    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [31]:
# Here we go, answer the questions using the Merck Manual and RAG
rag_answer_1 = generate_rag_response(query_1)
print(rag_answer_1)

Thank you for your question! The Merck Manual provides comprehensive guidelines for managing sepsis in a critical care unit. According to Chapter 227, the protocol for managing sepsis includes the following steps:
1. First aid: Keep the patient warm, control hemorrhage, check airway and ventilation, and provide respiratory assistance if necessary.
2. Supplemental oxygen by face mask is provided, and the patient's head is turned to one side to avoid aspiration if emesis occurs.
3. Treatment begins simultaneously with evaluation


**RAG Response — Query 1 (Sepsis)**

Here we see the answer being drawn from the Merck manual, even giving the chapter number. The answer is concise.

When compared to the previous baseline query using Combo 4, the RAG answer is too short and only is pulling what it knows from the Merck manual.


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [32]:
rag_answer_2 = generate_rag_response(query_2)
print(rag_answer_2)

Thank you for reaching out! I'm happy to help you with your questions about appendicitis.
To answer your first question, the common symptoms of appendicitis include epigastric or periumbilical pain that shifts to the right lower quadrant after a few hours, nausea, vomiting, and anorexia. These symptoms typically develop rapidly over several hours, and the pain often increases with coughing or movement.
Unfortunately, appendicitis is not curable via medicine alone. The condition requires surgical intervention to remove the inflamed appendix, which


**RAG Response — Query 2 (Appendicitis)**

This RAG response is much more detailed than the baseline, including the sympton movement pattern, and the fact that surgical intervention is needed.



### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [33]:
rag_answer_3 = generate_rag_response(query_3, k=5, max_tokens=256)
print(rag_answer_3)

The effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, depend on the underlying cause of the condition. Here are some possible causes and their associated treatments based on the information provided in the Merck Manual:
1. Alopecia Areata: This is a common autoimmune disorder that causes sudden patchy hair loss. Treatment options for alopecia areata include topical corticosteroids, immunotherapy (diphencyprone or squaric acid dibutylester), or systemic corticosteroids. In severe cases, minoxidil or anthralin may be prescribed.
2. Traction Alopecia: This condition is caused by physical traction or stress on the scalp, leading to hair loss. Treatment involves eliminating the causative factor, such as tight hairstyles or hair extensions.
3. Tinea Capitis: This is a fungal infection that causes hair loss. Treatment involves topical or oral antifungals, such as terbinafine or griseofulvin.
4. Trichotill


**RAG Response — Query 3 (Hair Loss) **

Much better answer than the baseline, which blamed alopcecia (male pattern baldness), as the root cause, and did not include the autoimmune issue. The system asking us for more details about our specific condition is really good, and then starts to go into some general information. Please note, the change to k=5 is a result of my fine tuning (below - after the queries are run), and I cam back and changed this. Leaving k=3 means that autoimmune will not be picked up in the answer.



### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [34]:
rag_answer_4 = generate_rag_response(query_4)
print(rag_answer_4)

Based on the information provided in the Merck Manual, there are several treatments recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function. These include:
1. Supportive care: Providing good nutrition, preventing systemic complications due to immobilization (such as pneumonia, UTI, thromboembolic disease), and preventing pressure ulcers are important aspects of supportive care for patients with brain injury.
2. Physical therapy: Physical therapy may modestly improve


**RAG Response — Query 4 (Brain Injury)**

Strong, confident answer with specific clinical detail not present in the earlier baseline. This is just a much better answer for a serious condition.



### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [35]:
rag_answer_5 = generate_rag_response(query_5)
print(rag_answer_5)

As a knowledgeable medical assistant, I can provide guidance on the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip. However, please note that I am only providing general information based on the context provided in the Merck Manual, and not as a substitute for professional medical advice.
For a person with a fractured leg, it is essential to follow these precautions:
1. Immobilization: The affected limb should be immobilized using a splint or cast to prevent further injury and promote healing. This will help reduce


**RAG Response — Query 5 (Leg Fracture)**

On this one I liked the disclaimer it added. This just read so much better than the base line. Also, no need to manipulate the temperature.


**Section Summary — RAG Question Answering**

Across the 5 queries, RAG grounding showed mixed but good, informative results

Query 1 had the chapter noted.

Query 2 was good with specificity and clinical detail.

Query 3 - well this was a tough question to begin with. We can put a man on the moon but we can't stop hair loss.....we will fine tune this one

Query 4 and 5 - both good with specificty and detail.



### Fine-tuning

In [36]:
# Query 3 - lets fine tune =  give us five combinations and see what comes out - takes about 3 minutes to run
query_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

finetune_combinations = [
    {"label": "1. Baseline (k=3, temp=0)", "k": 3, "max_tokens": 128, "temperature": 0, "top_p": 0.95, "top_k": 50},
    {"label": "2. Higher k=5 (more context)", "k": 5, "max_tokens": 128, "temperature": 0, "top_p": 0.95, "top_k": 50},
    {"label": "3. Lower k=1 (most focused chunk only)", "k": 1, "max_tokens": 128, "temperature": 0, "top_p": 0.95, "top_k": 50},
    {"label": "4. k=5, longer output", "k": 5, "max_tokens": 256, "temperature": 0, "top_p": 0.95, "top_k": 50},
    {"label": "5. k=5, slight temperature", "k": 5, "max_tokens": 256, "temperature": 0.3, "top_p": 0.9, "top_k": 40},
]

finetune_results = {}
for combo in finetune_combinations:
    answer = generate_rag_response(
        query_3,
        k=combo["k"],
        max_tokens=combo["max_tokens"],
        temperature=combo["temperature"],
        top_p=combo["top_p"],
        top_k=combo["top_k"]
    )
    finetune_results[combo["label"]] = answer
    print(f"=== {combo['label']} ===")
    print(answer)
    print()

=== 1. Baseline (k=3, temp=0) ===
Thank you for reaching out! The information provided in the Merck Manual can be helpful in understanding the different types of alopecia and their potential treatments. However, I must inform you that I cannot provide a definitive answer to your question without more context and information about your specific situation.
Sudden patchy hair loss, also known as localized bald spots on the scalp, can have various causes, including:
1. Androgenetic alopecia: This is the most common cause of sudden patchy hair loss in both men and women. It is an androgen-dependent her

=== 2. Higher k=5 (more context) ===
The effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, depend on the underlying cause of the condition. Here are some possible causes and their associated treatments based on the information provided in the Merck Manual:
1. Alopecia Areata: This is a common autoimmune disorder that

**Fine-Tuning Results — Query 3 (Hair Loss)**

Each one of these results showed how the system is generating the answers.

With label 4, the longer output with k=5 and more tokens, we get the autoimmune answer that we are seeking (because it is buried a little deeper in the chunked data). Label 5 increases the temperature a bit, but it gets a bit too wordy in my opinion.

So of the three, Combo 4 was the best one, most information and clinical details.



## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answers generated to the questions from the previous section.

- We are using the same LLaMA model for evaluation, so basically here the LLM is rating itself on how well it has performed in the task.

*Pretty important to note that - it is rating itself - so not an impartial judge, but will do the job for what we are working on here.

In [37]:
groundedness_rater_system_message = """You are an expert evaluator assessing whether an AI-generated answer is grounded in the provided context.
Groundedness means the answer only contains information that can be directly supported by the context — it does not add facts, figures, or claims that are not present in the context.

Rate the groundedness of the answer on a scale of 1 to 5, where:
1 = Not grounded at all (answer contradicts or ignores the context, or is entirely fabricated)
3 = Partially grounded (some claims supported by context, others not)
5 = Fully grounded (every claim in the answer is directly supported by the context)

Provide your rating as a single number, followed by a brief one-sentence justification."""

In [38]:
relevance_rater_system_message = """You are an expert evaluator assessing whether an AI-generated answer is relevant to the question asked.
Relevance means the answer directly addresses what was asked, without going off-topic or answering a different question.

Rate the relevance of the answer on a scale of 1 to 5, where:
1 = Not relevant at all (answer does not address the question)
3 = Partially relevant (answer addresses some but not all parts of the question)
5 = Fully relevant (answer directly and completely addresses the question asked)

Provide your rating as a single number, followed by a brief one-sentence justification."""

In [39]:
user_message_template = """Context:
{context}

Question: {question}

Answer to evaluate: {answer}

Please provide your rating and justification:"""

In [40]:
# new function to judge the relevance
def generate_ground_relevance_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    dynamic_retriever = vectordb.as_retriever(search_type="similarity", search_kwargs={"k": k})
    relevant_document_chunks = dynamic_retriever.invoke(user_input)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    prompt = f"[INST] {qna_system_message}\n{qna_user_message_template.format(context=context_for_query, question=user_input)} [/INST]"

    llm.reset()
    response = llm(prompt=prompt, max_tokens=max_tokens, temperature=temperature, top_p=top_p, top_k=top_k, stop=['INST'])
    answer = response["choices"][0]["text"]

    groundedness_prompt = f"[INST] {groundedness_rater_system_message}\n{user_message_template.format(context=context_for_query, question=user_input, answer=answer)} [/INST]"
    llm.reset()
    response_1 = llm(prompt=groundedness_prompt, max_tokens=max_tokens, temperature=temperature, top_p=top_p, top_k=top_k, stop=['INST'])

    relevance_prompt = f"[INST] {relevance_rater_system_message}\n{user_message_template.format(context=context_for_query, question=user_input, answer=answer)} [/INST]"
    llm.reset()
    response_2 = llm(prompt=relevance_prompt, max_tokens=max_tokens, temperature=temperature, top_p=top_p, top_k=top_k, stop=['INST'])

    return answer, response_1['choices'][0]['text'], response_2['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

Note: these queries take about 3 min each to run.

In [41]:
answer, groundedness, relevance = generate_ground_relevance_response(query_1, k=5)
print("ANSWER:\n", answer)
print("\nGROUNDEDNESS RATING:\n", groundedness)
print("\nRELEVANCE RATING:\n", relevance)

ANSWER:
   Thank you for your question! The management of sepsis in a critical care unit is a complex and multifaceted process that involves several steps. Here are some key elements of the protocol for managing sepsis in a critical care unit, based on the information provided in the Merck Manual:
1. Initial Assessment: The first step in managing sepsis is to quickly assess the patient's vital signs, including temperature, heart rate, respiratory rate, and blood pressure. This will help identify any signs of organ dysfunction or failure.
2. Fluid Res

GROUNDEDNESS RATING:
   Based on the information provided in the passage, I would rate the groundedness of the answer as a 4 out of 5. Here's my reasoning:
The answer provides specific and accurate information related to the management of sepsis in a critical care unit, including the initial assessment, fluid resuscitation, antibiotics, surgical excision of infected tissues, and supportive care. The answer also references specific chapter

Interestingly it gave a 4 rating, I probably would have been at 3

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [42]:
answer, groundedness, relevance = generate_ground_relevance_response(query_2, k=5)
print("ANSWER:\n", answer)
print("\nGROUNDEDNESS RATING:\n", groundedness)
print("\nRELEVANCE RATING:\n", relevance)

ANSWER:
   Thank you for reaching out! I'm happy to help you with your questions about appendicitis.
Common Symptoms of Appendicitis:
The common symptoms of appendicitis include:
1. Epigastric or periumbilical pain that begins suddenly and worsens over time, often followed by nausea, vomiting, and anorexia.
2. Right lower quadrant tenderness, especially after palpation at McBurney's point (the junction of the middle and outer thirds of the line joining the umbilicus to the anterior

GROUNDEDNESS RATING:
   Based on the provided context, I would rate the groundedness of the answer as a 4 out of 5. Here's my justification:
The answer provides a clear and concise explanation of the common symptoms of appendicitis, which are well-supported by the context. The answer accurately lists the classic symptoms of appendicitis, including epigastric or periumbilical pain that worsens over time, nausea, vomiting, and anorexia. It also mentions that these symptoms are typically accompanied by right l

Good reply, and I agree this would be a 4.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [43]:
answer, groundedness, relevance = generate_ground_relevance_response(query_3, k=5)
print("ANSWER:\n", answer)
print("\nGROUNDEDNESS RATING:\n", groundedness)
print("\nRELEVANCE RATING:\n", relevance)

ANSWER:
   The effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, depend on the underlying cause of the condition. Here are some possible causes and their associated treatments based on the information provided in the Merck Manual:
1. Alopecia Areata: This is a common autoimmune disorder that causes sudden patchy hair loss. Treatment options for alopecia areata include topical corticosteroids, immunotherapy (diphencyprone or squaric acid dibutylester), or systemic

GROUNDEDNESS RATING:
   Based on the information provided in the passage, I would rate the groundedness of the answer as a 4 out of 5. Here's my justification:
The answer provides a comprehensive overview of the different types of hair disorders, their causes, and treatment options. It mentions various conditions such as alopecia areata, hyperandrogenemia, tinea capitis, trichotillomania, and traction alopecia, among others. The answer also explains t

I am in agreement

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [44]:
answer, groundedness, relevance = generate_ground_relevance_response(query_4, k=5)
print("ANSWER:\n", answer)
print("\nGROUNDEDNESS RATING:\n", groundedness)
print("\nRELEVANCE RATING:\n", relevance)

ANSWER:
   Based on the information provided in the Merck Manual of Diagnosis & Therapy, the recommended treatments for a person who has sustained a physical injury to brain tissue resulting in temporary or permanent impairment of brain function are:
1. Supportive care: This includes preventing systemic complications due to immobilization, providing good nutrition, and preventing pressure ulcers.
2. Physical and occupational therapy: These therapies may modestly improve functioning but are more important for making the environment safer and providing devices that help patients circumvent the primary def

GROUNDEDNESS RATING:
   Based on the information provided in the passage, I would rate the groundedness of the answer as a 5, fully grounded. The answer provides specific and directly supported claims about the recommended treatments for a person who has sustained a physical injury to brain tissue resulting in temporary or permanent impairment of brain function.
The answer mentions sup

Yes, a groundness of 5 is justified here.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [45]:
answer, groundedness, relevance = generate_ground_relevance_response(query_5, k=5)
print("ANSWER:\n", answer)
print("\nGROUNDEDNESS RATING:\n", groundedness)
print("\nRELEVANCE RATING:\n", relevance)

ANSWER:
   As a knowledgeable medical assistant, I can provide you with information based on the context provided in the Merck Manual of Diagnosis & Therapy.
For a person who has fractured their leg during a hiking trip, the necessary precautions and treatment steps include:
Precautions:
1. Immobilization: The affected leg should be immobilized using a splint or cast to prevent further injury and promote healing.
2. Rest: The patient should rest the affected leg and avoid putting weight on it as much as possible to reduce pain and prevent further damage.


GROUNDEDNESS RATING:
   Based on the provided context, I would rate the groundedness of the answer as a 5, fully grounded. The answer provides specific and directly supported claims related to the treatment and care of a person who has fractured their leg during a hiking trip.
1. Immobilization: The answer correctly states that immobilization is necessary to prevent further injury and promote healing (Grounded, 5).
2. Rest: The answe

Another 5 and I agree.

**Section Summary — Output Evaluation**

All 5 queries scored 4-5 out of 5 on both groundedness and relevance, using the LLM itself as judge. I would have given Q1 a 3.

Q 4 and 5 each got a perfect score.

Now, the same LLaMA model is evaluating itself, so take this with a grain of salt.


## Actionable Insights and Business Recommendations

### Key Findings

RAG improves the specificity with domain knowledge from the Merck manual, and we can see this when compared against the answers from the LLM baseline.

We needed to use k-5 instead of default k=3 to dig deeper into the answer, especially for query 3 and hair loss.

The prompt formatting is important, as we saw with the baseline queries (2 and 5) which came back with empty results due to formatting issues in the prompt.

Temperature tuning is another area that peaked interest and adjustment, as higher temperature (0.7) caused bad outputs on the procedural and first aid questions.

The self evaluation tool has a credibility issue, it is judging itself, so this is not a good independant method to evaluate.

Token length - to save space, this notebook used 256 tokens, and we can see that the answers were truncated. In real life, we will want the full responses, so using the max tokens 512 is recommended.


### Business Recommendations

Deploy RAG over the stand alone LLM - this will give us domain specific information (Merck manual) over top of our LLM's data.

We should do more retrieval tuning (k, chuncks, embedding), as we saw with Query 3's fine tuning and adjusting k to 5, it came back with the correct response, whereas using the default k=3, it was not the complete answer. A medical person needs to be involved here, as my medical knowledge is limited.

We should not assume the self-evaluation scores will hold up to an audit or be used in production, they are fine for our testing and proving concepts, but for production, we need to compare with an independant evaluation.

There were a number of panda, numpy, langchain and other technical issues, which caused so much delay in building this,  meaning that this is a fragile system that really needs hardened prior to going to production.



### Potential Business Impact

This system, once vetted, will help our care team by giving Merck Manual domain knowledge over top of our generic LLM answers, and these deep, generated responses can be used for guidance, as well as it can be used as a training and educational aid.



<font size=6 color='blue'>Power Ahead</font>
___

In [46]:
!jupyter nbconvert --to html '/content/drive/MyDrive/Colab Notebooks/MedicalAssistant_RAG_LLaMA_FullCode_v5.ipynb'

[NbConvertApp] WARNING | pattern '/content/drive/MyDrive/Colab Notebooks/MedicalAssistant_RAG_LLaMA_FullCode_v5.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of promptin